In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql import functions as F

movies_df = spark.read.table("movie_silver.movies") \
                        .filter(f"file_date = '{v_file_date}'")  
countries_df = spark.read.table("movie_silver.countries")
movies_countries_df = spark.read.table("movie_silver.productions_countries") \
                        .filter(f"file_date = '{v_file_date}'")  
companies_df = spark.read.table("movie_silver.productions_companies")
movies_companies_df = spark.read.table("movie_silver.movies_companies") \
                        .filter(f"file_date = '{v_file_date}'")  

In [0]:
result_df = (
    movies_df.alias("m")
    .filter(F.col("year_release_date") >= 2010)
    .join(
        movies_countries_df.alias("mc"),
        F.col("m.movie_id") == F.col("mc.movie_id"),
        "inner"
    )
    .join(
        countries_df.alias("c"),
        F.col("mc.country_id") == F.col("c.country_id"),
        "inner"
    )
    .join(
        movies_companies_df.alias("mcn"),
        F.col("m.movie_id") == F.col("mcn.movie_id"),
        "inner"
    )
    .join(
        companies_df.alias("cn"),
        F.col("mcn.company_id") == F.col("cn.company_id"),
        "inner"
    )
    .select(
        F.col("m.movie_id"),
        F.col("c.country_id"),
        F.col("cn.company_id"),
        F.col("m.title"),
        F.col("m.budget"),
        F.col("m.revenue"),
        F.col("m.duration_time"),
        F.col("m.release_date"),
        F.col("c.country_name"),
        F.col("cn.company_name")
    )
    .withColumn("created_date", F.lit(v_file_date))
    .orderBy(F.asc("title"))
)

In [0]:
#overwrite_partition("movie_gold","results_country_prod_company","created_date", v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date = src.created_date"
merge_delta_lake(result_df,"movie_gold","results_country_prod_company",merge_condition,"created_date")

In [0]:
%sql
SELECT created_date,
COUNT(1) 
FROM movie_gold.results_country_prod_company
GROUP BY created_date

created_date,count(1)
2024-12-16,95
2024-12-23,1548
2024-12-30,2352
